# 감정 분류 klue/bert-base 파인튜닝 (Colab GPU용)

**목표**: AI-Hub + KOTE 데이터를 합쳐서 klue/bert-base로 4종 감정 분류 파인튜닝 (목표 정확도 70%+)

**라벨**: 기쁨, 슬픔, 화남, 두려움/놀라움 (+ "없음"은 예측 확률 임계값으로 별도 처리)

**실행 전 확인**: 상단 메뉴 `런타임 → 런타임 유형 변경 → 하드웨어 가속기: GPU(T4)` 선택 필수!

**흐름**
1. 환경 설정 (패키지 설치, GPU 확인)
2. 데이터 업로드 (AI-Hub JSON + KOTE tsv를 zip으로 업로드)
3. AI-Hub 로드
4. KOTE 로드 + 라벨 매핑
5. 두 데이터셋 합치기 + 4종 라벨 통합
6. train/test 분할 (BERT는 언더샘플링 대신 class weight로 불균형 처리)
7. 토크나이징
8. klue/bert-base 파인튜닝
9. 평가 (classification report + confusion matrix)
10. 모델 저장 (Google Drive에 zip으로)
11. 빠른 테스트 (threshold 기반 "없음" 처리)


## 1. 환경 설정

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas numpy
import torch
print("GPU 사용 가능:", torch.cuda.is_available())
print("디바이스명:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU만 사용 중 - 런타임 유형을 GPU로 바꿔주세요!")


## 2. 데이터 업로드

로컬 `AI_project` 폴더에서 아래 파일/폴더들을 하나의 zip으로 압축해서 업로드해줘:
- `01.원천데이터` (AI-Hub 성인/청소년 라벨링 JSON들이 들어있는 폴더, 폴더명은 실제 구조에 맞게 아래 경로 수정)
- `data/train.tsv`, `data/test.tsv`, `data/val.tsv` (KOTE)

압축 파일명은 아무거나 상관없음 (예: `data_upload.zip`). 아래 셀 실행하면 업로드 창이 뜸.


In [ ]:
from google.colab import files
import zipfile
import os

uploaded = files.upload()  # zip 파일 선택해서 업로드

for fname in uploaded.keys():
    if fname.endswith(".zip"):
        with zipfile.ZipFile(fname, "r") as z:
            z.extractall(".")
        print(f"{fname} 압축 해제 완료")

# 압축 해제 후 폴더 구조 확인 (경로가 예상과 다르면 아래 AIHUB_DATA_DIRS, kote_files 경로를 맞게 수정)
!find . -maxdepth 3 -type d | grep -v "^\./\." | grep -v sample_data


## 3. AI-Hub 데이터 로드

In [ ]:
import json
import glob

# 실제 압축 해제된 경로에 맞게 수정 (위 find 결과 참고)
AIHUB_DATA_DIRS = [
    "data/성인라벨링데이터",
    "data/청소년라벨링데이터",
]

def load_aihub_json(data_dirs):
    records = []
    json_files = []
    for d in data_dirs:
        found = glob.glob(os.path.join(d, "**", "*.json"), recursive=True)
        print(f"{d}: {len(found)}개 파일 발견")
        json_files += found

    print(f"AI-Hub 전체 JSON 파일 수: {len(json_files)}")

    for filepath in json_files:
        try:
            with open(filepath, encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            print(f"파일 읽기 실패: {filepath} ({e})")
            continue

        for turn in data.get("Conversation", []):
            text = turn.get("Text", "").strip()
            emotion = turn.get("VerifyEmotionTarget", "").strip()
            if not text or not emotion:
                continue
            records.append({"text": text, "emotion": emotion, "source": "aihub"})

    return pd.DataFrame(records)

import pandas as pd
df_aihub = load_aihub_json(AIHUB_DATA_DIRS)
print(f"AI-Hub 문장 수: {len(df_aihub)}")
print(df_aihub["emotion"].value_counts())


## 4. KOTE 데이터 로드 + 라벨 매핑

KOTE는 44개 세분화 감정이 멀티라벨로 붙어있음. 우리 7종 라벨에 대응되는 것만 매핑.


In [ ]:
kote_label_names = ['불평/불만', '환영/호의', '감동/감탄', '지긋지긋', '고마움', '슬픔', '화남/분노', '존경',
                     '기대감', '우쭐댐/무시함', '안타까움/실망', '비장함', '의심/불신', '뿌듯함', '편안/쾌적',
                     '신기함/관심', '아껴주는', '부끄러움', '공포/무서움', '절망', '한심함', '역겨움/징그러움',
                     '짜증', '어이없음', '없음', '패배/자기혐오', '귀찮음', '힘듦/지침', '즐거움/신남', '깨달음',
                     '죄책감', '증오/혐오', '흐뭇함(귀여움/예쁨)', '당황/난처', '경악', '부담/안_내킴', '서러움',
                     '재미없음', '불쌍함/연민', '놀람', '행복', '불안/걱정', '기쁨', '안심/신뢰']

assert len(kote_label_names) == 44, "라벨 개수가 44개가 아닙니다."

kote_to_ours = {
    "기쁨": "기쁨", "행복": "기쁨", "즐거움/신남": "기쁨", "뿌듯함": "기쁨",
    "흐뭇함(귀여움/예쁨)": "기쁨", "감동/감탄": "기쁨",
    "없음": "없음",
    "놀람": "놀라움", "경악": "놀라움", "당황/난처": "놀라움", "신기함/관심": "놀라움",
    "아껴주는": "사랑스러움", "고마움": "사랑스러움", "존경": "사랑스러움",
    "안심/신뢰": "사랑스러움", "환영/호의": "사랑스러움", "편안/쾌적": "사랑스러움",
    "화남/분노": "화남", "짜증": "화남", "지긋지긋": "화남", "역겨움/징그러움": "화남",
    "증오/혐오": "화남", "어이없음": "화남", "한심함": "화남", "우쭐댐/무시함": "화남",
    "공포/무서움": "두려움", "불안/걱정": "두려움", "부담/안_내킴": "두려움",
    "슬픔": "슬픔", "절망": "슬픔", "서러움": "슬픔", "패배/자기혐오": "슬픔",
    "죄책감": "슬픔", "힘듦/지침": "슬픔", "안타까움/실망": "슬픔", "불쌍함/연민": "슬픔",
}

kote_files = ["data/train.tsv", "data/test.tsv", "data/val.tsv"]
df_kote_raw = pd.concat([
    pd.read_csv(f, sep="\t", header=None, names=["id", "text", "labels"])
    for f in kote_files
], ignore_index=True)

print(f"KOTE 원본 문장 수: {len(df_kote_raw)}")

records = []
for _, row in df_kote_raw.iterrows():
    text = str(row["text"]).strip()
    if not text or pd.isna(row["labels"]):
        continue

    idxs = [int(x) for x in str(row["labels"]).split(",")]
    mapped = set()
    for idx in idxs:
        if idx < len(kote_label_names):
            name = kote_label_names[idx]
            if name in kote_to_ours:
                mapped.add(kote_to_ours[name])

    for emo in mapped:
        records.append({"text": text, "emotion": emo, "source": "kote"})

df_kote = pd.DataFrame(records)
print(f"KOTE 매핑 후 문장 수: {len(df_kote)}")
print(df_kote["emotion"].value_counts())


## 5. 두 데이터셋 합치기 + 4종 라벨 통합

In [ ]:
df = pd.concat([df_aihub, df_kote], ignore_index=True)
df = df.drop_duplicates(subset=["text", "emotion"]).reset_index(drop=True)

print(f"합친 후 총 문장 수: {len(df)}")
print(df["source"].value_counts())

os.makedirs("data", exist_ok=True)
df.to_csv("data/emotion_dataset_merged.csv", index=False, encoding="utf-8-sig")
print("저장 완료: data/emotion_dataset_merged.csv")


In [ ]:
# 7종 -> 4종 통합, "없음"은 별도 취급 (학습에는 4종만 사용, 추론 시 확률 임계값으로 "없음" 처리)
emotion_map = {
    "기쁨": "기쁨",
    "사랑스러움": "기쁨",
    "슬픔": "슬픔",
    "화남": "화남",
    "두려움": "두려움/놀라움",
    "놀라움": "두려움/놀라움",
}
df_train_ready = df[df["emotion"] != "없음"].copy()
df_train_ready["emotion"] = df_train_ready["emotion"].map(emotion_map)
df_train_ready = df_train_ready.dropna(subset=["emotion"]).reset_index(drop=True)

print(df_train_ready["emotion"].value_counts())

label_list = sorted(df_train_ready["emotion"].unique())
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}
print(label2id)


## 6. train/test 분할

**언더샘플링 대신 class weight 사용**: 로지스틱회귀 baseline은 데이터 손실을 감수하고 언더샘플링했지만,
BERT는 데이터가 많을수록 유리하므로 전체 데이터를 다 쓰고 대신 손실함수에 클래스 가중치를 줘서 불균형을 보정함.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

df_train_ready["label"] = df_train_ready["emotion"].map(label2id)

train_df, test_df = train_test_split(
    df_train_ready, test_size=0.2, random_state=42, stratify=df_train_ready["label"]
)

print(f"학습 데이터: {len(train_df)}, 테스트 데이터: {len(test_df)}")

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(sorted(label2id.values())),
    y=train_df["label"].values,
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print("클래스 가중치:", dict(zip(label_list, class_weights.tolist())))


## 7. 토크나이징

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_pandas(train_df[["text", "label"]].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[["text", "label"]].reset_index(drop=True))

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=64)

train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

train_ds = train_ds.remove_columns(["text"])
test_ds = test_ds.remove_columns(["text"])
train_ds.set_format("torch")
test_ds.set_format("torch")


## 8. klue/bert-base 파인튜닝

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch.nn as nn

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_list), id2label=id2label, label2id=label2id
)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
)

from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro"),
    }

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

trainer.train()


## 9. 평가

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

preds_output = trainer.predict(test_ds)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

print(classification_report(y_true, y_pred, target_names=label_list))

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=label_list, columns=label_list)
cm_df


## 10. 모델 저장 (Google Drive)

Drive에 저장해두면 Colab 세션 끊겨도 안전함. 로컬 PC로 다운받아서 FastAPI 백엔드에서 쓰려면
저장된 폴더를 zip으로 묶어서 다운로드하면 됨.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/AI_project/emotion_bert_model"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

import json as _json
with open(os.path.join(SAVE_DIR, "label2id.json"), "w", encoding="utf-8") as f:
    _json.dump(label2id, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {SAVE_DIR}")


In [ ]:
# 로컬 다운로드용 zip 생성 (Drive 대신 바로 다운받고 싶으면 이 셀 사용)
!zip -r emotion_bert_model.zip /content/drive/MyDrive/AI_project/emotion_bert_model
from google.colab import files
files.download("emotion_bert_model.zip")


## 11. 빠른 테스트

TF-IDF 버전과 동일하게, 최고 확률이 threshold보다 낮으면 "없음"으로 처리.


In [ ]:
import torch.nn.functional as F

def predict_emotion_bert(text, threshold=0.5):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=64)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = F.softmax(logits, dim=-1)[0].cpu().numpy()
    prob_dict = {id2label[i]: float(p) for i, p in enumerate(probs)}

    max_prob = max(probs)
    if max_prob < threshold:
        pred = "없음"
    else:
        pred = id2label[int(np.argmax(probs))]

    return pred, prob_dict

for sample in ["슬퍼.", "너무 무서워서 도망치고 싶어", "완전 짜증나 진짜", "오늘 최고로 행복한 날이야"]:
    pred, probs = predict_emotion_bert(sample)
    print(f"입력: {sample}")
    print(f"예측: {pred}")
    print(f"확률: {probs}")
    print()
